# Environmental Risk Module — Gulf of Mannar / Palk Bay (Colab)

Original notebook by **Anannya Jadhav** (branch `environmental-risk-module`, notebook
`marine_pollution_prediction.ipynb`). This version keeps only the parts that hold up:

| Kept | Why |
|---|---|
| Copernicus Marine currents, SST, wave height (60 days, ~9 km grid, 245 sites) | Real model data; the main project had the loader but no login |
| Allen Coral Atlas benthic habitat per site (coral/algae, seagrass, habitat sensitivity, distance to coral) | Real mapped habitat; not available elsewhere in the project |
| Habitat-exposure index = sqrt(habitat value x physical exposure) | Transparent expert formula |
| Weight-sensitivity test (500 random weightings) | Shows how much the ranking depends on the chosen weights |

| Removed | Why |
|---|---|
| "Marine Pollution Impact Score" | Contained no pollution data (`plastic_risk` was 0 everywhere) |
| OBIS species-richness features and XGBoost/RF richness models | Richness mostly reflects survey effort (115/245 sites had 0 records); the main project already has an OBIS model with effort controls and spatial-block CV |
| `sensitive_species` | IUCN field came back empty for all 5,254 records (always 0) |
| SHAP-importance-as-weights and "biodiversity deficit" | Predictive importance is not ecological value; deficit used in-sample residuals of the same model |
| Shuffled K-fold CV | Sites 9 km apart with 10 km radius features overlap -> leaky |
| git clone / commit cells, `.gitignore` rewrite | Not part of the analysis; the rewrite would un-ignore `.venv`, datasets and weights |

Teammate's run (2026-10-14): 245 sites, 60 days (2026-08-04 -> 2026-10-02), grid cell ~9.2 km, mean current
0.122 m/s, mean SST 29.02 C, mean wave height 1.28 m; 106/245 sites have Coral/Algae and 96/245 seagrass within 10 km;
ranking Spearman vs default weights over 500 random weightings: mean 0.991 (min 0.877) - with the old value weights,
re-run for the numbers of this version.

**Needs:** a free Copernicus Marine account (asked for at runtime) and the Allen Coral Atlas benthic GeoJSON for the
area in Google Drive (`ATLAS_GEOJSON`). Note: the Atlas class is **"Coral/Algae"**, so `coral_cover_frac` is coral
*and* algae cover.

In [ ]:
!pip -q install copernicusmarine

In [ ]:
import os, json, math, time, warnings, getpass
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

BASE   = "/content/drive/MyDrive/Marine-Plastic-AI"
RAW    = f"{BASE}/data/raw"
PROC   = f"{BASE}/data/processed"
FIG    = f"{BASE}/outputs/figures"
TAB    = f"{BASE}/outputs/tables"
MODELS = f"{BASE}/models"
for p in [f"{RAW}/coral_atlas", f"{RAW}/copernicus", PROC, FIG, TAB, MODELS]:
    os.makedirs(p, exist_ok=True)

SEED = 42

POLY = [(78.356112, 9.17125),    # west
        (80.06426, 10.021422),   # north
        (80.419466, 9.335989),   # east
        (79.818754, 7.719526)]   # south
_lons, _lats = zip(*POLY)
PAD = 0.1
BBOX = dict(lon_min=min(_lons) - PAD, lon_max=max(_lons) + PAD,
            lat_min=min(_lats) - PAD, lat_max=max(_lats) + PAD)

RADIUS_KM = 10

END_DATE   = pd.Timestamp.today().normalize() - pd.Timedelta(days=12)   # analysis data lags a few days
N_DAYS     = 60
START_DATE = END_DATE - pd.Timedelta(days=N_DAYS - 1)

# Allen Coral Atlas benthic map exported for the polygon above (Atlas "Download data" -> GeoJSON)
ATLAS_GEOJSON = "/content/drive/MyDrive/coral atlas/Area-20260919192616/Benthic-Map/benthic.geojson"

def savefig(name):
    plt.savefig(f"{FIG}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

print("Study window:", START_DATE.date(), "->", END_DATE.date())
print("Download box:", {k: round(v, 2) for k, v in BBOX.items()})

## Copernicus Marine: currents, sea temperature, wave height

In [ ]:
import xarray as xr
import copernicusmarine as cm

print("copernicusmarine version:", cm.__version__)
cm.login(username=input("Copernicus username: "),
         password=getpass.getpass("Copernicus password: "),
         force_overwrite=True)

IDS = dict(
    cur    = ("cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",    ["uo", "vo"], True),
    thetao = ("cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m", ["thetao"],   True),
    wav    = ("cmems_mod_glo_wav_anfc_0.083deg_PT3H-i",       ["VHM0"],     False),
)

def fetch(name):
    dsid, variables, has_depth = IDS[name]
    path = f"{RAW}/copernicus/{name}_{START_DATE.date()}_{END_DATE.date()}.nc"
    if os.path.exists(path):
        print(f"{name}: loaded from Drive cache")
        return xr.open_dataset(path).load()
    print(f"{name}: downloading ...")
    kw = dict(dataset_id=dsid, variables=variables,
              minimum_longitude=BBOX["lon_min"], maximum_longitude=BBOX["lon_max"],
              minimum_latitude=BBOX["lat_min"],  maximum_latitude=BBOX["lat_max"],
              start_datetime=str(START_DATE.date()), end_datetime=f"{END_DATE.date()}T23:59:59")
    if has_depth:
        kw.update(minimum_depth=0, maximum_depth=1)
    ds = cm.open_dataset(**kw).load()
    if "depth" in ds.dims:
        ds = ds.isel(depth=0, drop=True)
    ds.to_netcdf(path)
    return ds

data = {}
for name in ["cur", "thetao", "wav"]:
    try:
        data[name] = fetch(name)
        print(f"  OK  {name}: {dict(data[name].sizes)}")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {str(e)[:500]}")

if len(data) == 3:
    cur, thetao, wav = data["cur"], data["thetao"], data["wav"]
    print("\nAll three datasets ready. Go on to Cell 4.")

## Align to a daily grid and define the study sites

In [ ]:
from sklearn.neighbors import BallTree
from matplotlib.path import Path as MPath

# ---- align everything to daily values on the currents grid ----
wav = wav.resample(time="1D").mean()                  # 3-hourly waves -> daily mean
day = lambda ds: ds.assign_coords(time=ds["time"].dt.floor("D"))
cur, thetao, wav = day(cur), day(thetao), day(wav)
thetao = thetao.interp(latitude=cur.latitude, longitude=cur.longitude, method="nearest").reindex(time=cur.time, method="nearest")
wav    = wav.interp(latitude=cur.latitude, longitude=cur.longitude, method="nearest").reindex(time=cur.time, method="nearest")

order = ("time", "latitude", "longitude")
uo, vo = cur.uo.transpose(*order), cur.vo.transpose(*order)
th, hs = thetao.thetao.transpose(*order), wav.VHM0.transpose(*order)

# ---- study sites = ocean cells with valid currents AND inside your Atlas polygon ----
valid  = (uo.notnull().mean("time") > 0.9).values
iy, ix = np.where(valid)
lat_c, lon_c = cur.latitude.values[iy], cur.longitude.values[ix]
inside = MPath(POLY + [POLY[0]]).contains_points(np.c_[lon_c, lat_c])
iy, ix, lat_c, lon_c = iy[inside], ix[inside], lat_c[inside], lon_c[inside]

sites = pd.DataFrame({"site_id": [f"S{i:03d}" for i in range(len(iy))], "lat": lat_c, "lon": lon_c})
CELL_DEG = float(np.abs(np.diff(cur.latitude.values)).mean())
CELL_KM  = CELL_DEG * 111.0

U, V, T, W = (a.values[:, iy, ix] for a in (uo, vo, th, hs))      # shape [time, site]

def fill_missing(arr, sites):
    """Interpolate small gaps in time; sites with no data at all copy the nearest valid site."""
    arr = pd.DataFrame(arr).interpolate(limit_direction="both").values
    bad = np.isnan(arr).all(axis=0)
    if bad.any():
        good = np.where(~bad)[0]
        tree = BallTree(np.radians(sites.iloc[good][["lat", "lon"]].values), metric="haversine")
        _, j = tree.query(np.radians(sites.loc[bad, ["lat", "lon"]].values), k=1)
        arr[:, bad] = arr[:, good[j[:, 0]]]
    return arr

nan_share = {n: float(np.isnan(a).mean().round(3)) for n, a in zip("UVTW", (U, V, T, W))}
U, V, T, W = (fill_missing(a, sites) for a in (U, V, T, W))
SPEED = np.hypot(U, V)
dates = pd.to_datetime(cur.time.values)

print(f"{len(sites)} sites | {len(dates)} days | grid cell ≈ {CELL_KM:.1f} km")
print("share of missing values before filling:", nan_share)
print(f"mean current {SPEED.mean():.3f} m/s | mean SST {T.mean():.2f} °C | mean wave height {W.mean():.2f} m")

sites.to_csv(f"{PROC}/sites.csv", index=False)
np.savez_compressed(f"{PROC}/ocean_arrays.npz", U=U, V=V, T=T, W=W, dates=dates.astype(str).values)
json.dump(dict(cell_deg=CELL_DEG, cell_km=CELL_KM), open(f"{PROC}/grid_info.json", "w"))

# ---- sanity plot: sites inside the polygon ----
px, py = zip(*(POLY + [POLY[0]]))
plt.figure(figsize=(7, 7))
sc = plt.scatter(sites.lon, sites.lat, c=SPEED.mean(0), cmap="Blues", s=45, edgecolor="k", linewidth=.3)
plt.plot(px, py, "r--", label="Coral Atlas polygon"); plt.colorbar(sc, label="mean current speed (m/s)")
plt.legend(); plt.xlabel("lon"); plt.ylabel("lat"); plt.title(f"{len(sites)} study sites (ocean cells inside polygon)")
savefig("00_study_sites")

## Allen Coral Atlas habitat table

In [ ]:
import geopandas as gpd

LIGHT_PATH = f"{PROC}/coral_polygons_light.csv"

if os.path.exists(LIGHT_PATH):
    light = pd.read_csv(LIGHT_PATH)
    print("loaded cached habitat table from Drive")
else:
    assert os.path.exists(ATLAS_GEOJSON), f"Allen Coral Atlas benthic GeoJSON not found: {ATLAS_GEOJSON}"
    print("reading", ATLAS_GEOJSON, f"({os.path.getsize(ATLAS_GEOJSON)/1e6:.0f} MB) ... 1-3 minutes")
    g = gpd.read_file(ATLAS_GEOJSON)
    print("polygons:", len(g), "| columns:", list(g.columns), "| crs:", g.crs)
    CLASS_COL = next(c for c in ["class", "Class", "CLASS"] if c in g.columns)
    g = g.to_crs(32644)
    cen = g.geometry.centroid
    light = pd.DataFrame({"cls": g[CLASS_COL].values, "x": cen.x.values,
                          "y": cen.y.values, "area_m2": g.geometry.area.values})
    light.to_csv(LIGHT_PATH, index=False)
    del g, cen
    print("saved a compact habitat table to Drive")
print(light.cls.value_counts())

## Habitat features per site (within `RADIUS_KM`)

In [ ]:
from pyproj import Transformer
from scipy.spatial import cKDTree

# ---- sensitivity weight per habitat class (assumption; explain in report) ----
BENTHIC_SENS = {"coral": 1.0, "seagrass": 0.8, "rubble": 0.4, "rock": 0.3, "sand": 0.1}
def class_weight(name):
    n = str(name).lower()
    return next((w for k, w in BENTHIC_SENS.items() if k in n), 0.2)

light["w"] = light.cls.map(class_weight)
is_coral   = light.cls.str.contains("coral", case=False).values
is_seagras = light.cls.str.contains("seagrass", case=False).values
P    = light[["x", "y"]].values
area = light.area_m2.values
wts  = light.w.values

to_utm = Transformer.from_crs(4326, 32644, always_xy=True)
sx, sy = to_utm.transform(sites.lon.values, sites.lat.values)
S = np.c_[sx, sy]

R = RADIUS_KM * 1000
circle_area = math.pi * R ** 2
tree_all = cKDTree(P)
ind = tree_all.query_ball_point(S, r=R)                 # habitat polygons (by centroid) within the radius of each site

rows = []
for idx in ind:
    idx = np.asarray(idx, dtype=int)
    a = area[idx]
    rows.append(dict(mapped_frac=a.sum() / circle_area,
                     coral_cover_frac=a[is_coral[idx]].sum() / circle_area,
                     seagrass_frac=a[is_seagras[idx]].sum() / circle_area,
                     habitat_sens=(a * wts[idx]).sum() / circle_area))
coral_feats = pd.DataFrame(rows, index=sites.index)

d_coral, _ = cKDTree(P[is_coral]).query(S)                # distance to nearest Coral/Algae polygon
coral_feats["coral_dist_km"] = np.clip(d_coral / 1000, 0, 50)

print("sites with coral/algae inside the radius:", int((coral_feats.coral_cover_frac > 0).sum()), "of", len(sites))
print("sites with seagrass inside the radius   :", int((coral_feats.seagrass_frac > 0).sum()), "of", len(sites))
display(coral_feats.describe().round(3))
coral_feats.assign(site_id=sites.site_id).to_csv(f"{PROC}/coral_features.csv", index=False)

# ---- maps ----
to_ll = Transformer.from_crs(32644, 4326, always_xy=True)
samp = np.random.default_rng(SEED).choice(np.where(is_coral)[0], size=min(30000, is_coral.sum()), replace=False)
clon, clat = to_ll.transform(P[samp, 0], P[samp, 1])
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
for a_, col, cmap in zip(ax, ["coral_cover_frac", "habitat_sens", "coral_dist_km"], ["YlGn", "YlGn", "cividis_r"]):
    a_.scatter(clon, clat, s=0.3, c="lightgrey", zorder=0)                       # grey dots = mapped Coral/Algae
    s = a_.scatter(sites.lon, sites.lat, c=coral_feats[col], cmap=cmap, s=45, edgecolor="k", linewidth=.2, zorder=2)
    plt.colorbar(s, ax=a_); a_.set_title(col); a_.set_xlabel("lon"); a_.set_ylabel("lat")
plt.suptitle("Habitat features per site (grey = mapped Coral/Algae polygons)", fontsize=14)
savefig("02_habitat_maps")

## Master site table

In [ ]:
mean_dir = (np.degrees(np.arctan2(U.mean(0), V.mean(0))) + 360) % 360      # direction the water flows TOWARD (0 = north)
static = pd.concat([sites, coral_feats], axis=1)
static["current_speed"] = SPEED.mean(0)
static["current_dir"]   = mean_dir
static["temperature"]   = T.mean(0)
static["wave_height"]   = W.mean(0)
static.to_csv(f"{PROC}/site_features.csv", index=False)
print("master site table:", static.shape)
display(static.head())

# ---- quick look at every feature ----
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
for a_, col, cmap in zip(ax.ravel(),
        ["habitat_sens", "coral_cover_frac", "coral_dist_km",
         "current_speed", "wave_height", "temperature"],
        ["YlGn", "YlGn", "cividis_r", "Blues", "Purples", "coolwarm"]):
    s = a_.scatter(static.lon, static.lat, c=static[col], cmap=cmap, s=35); plt.colorbar(s, ax=a_)
    a_.set_title(col); a_.set_xlabel("lon"); a_.set_ylabel("lat")
plt.suptitle("Environmental feature maps (study sites)", fontsize=15); plt.tight_layout(); savefig("03_feature_maps")

num = static.drop(columns=["site_id", "lat", "lon"])
plt.figure(figsize=(10, 8))
plt.imshow(num.corr(), cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(num.shape[1]), num.columns, rotation=90); plt.yticks(range(num.shape[1]), num.columns)
plt.title("Feature correlation"); savefig("04_correlation")

## Habitat-exposure index
*Value* = how much sensitive habitat is near the site (Coral Atlas); *exposure* = physical transport/disturbance
(currents, waves). Multiplicative, so a site is high only if both are high. The weights are **assumptions**; the next
cell tests how much the ranking depends on them. No plastic data exists for this area yet, so this is **not** a
pollution score.

In [ ]:
def normalize(s):
    """0-1 scale; a constant column becomes all zeros rather than dividing by zero."""
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0

# ---- habitat value: how much sensitive habitat is near the site (assumed weights) ----
VALUE_W = {"habitat_sens": 0.5, "coral_cover_frac": 0.25, "seagrass_frac": 0.25}

# ---- physical exposure: proxies for pollutant transport/disturbance ----
EXPOSURE_W = {"current_speed": 0.5, "wave_height": 0.5}

value_norm    = pd.DataFrame({k: normalize(static[k]) for k in VALUE_W})
exposure_norm = pd.DataFrame({k: normalize(static[k]) for k in EXPOSURE_W})

value_score    = sum(value_norm[k] * w for k, w in VALUE_W.items())
exposure_score = sum(exposure_norm[k] * w for k, w in EXPOSURE_W.items())

# ---- multiplicative: a site only scores high risk if BOTH value and exposure are high ----
# tiny floor avoids a hard zero wiping out a site that's otherwise borderline
risk_score = np.sqrt((value_score + 1e-6) * (exposure_score + 1e-6))

risk = static[["site_id", "lat", "lon"]].copy()
risk["value_score"] = value_score
risk["exposure_score"] = exposure_score
risk["risk_score"] = risk_score
risk["risk_rank"] = risk_score.rank(ascending=False).astype(int)   # 1 = highest risk

risk.to_csv(f"{PROC}/habitat_exposure_index.csv", index=False)
print("top 10 highest-risk sites:")
display(risk.sort_values("risk_score", ascending=False).head(10))

plt.figure(figsize=(7, 7))
sc = plt.scatter(risk.lon, risk.lat, c=risk.risk_score, cmap="Reds", s=50, edgecolor="k", linewidth=.3)
plt.colorbar(sc, label="habitat-exposure index (0-1, geometric mean; no plastic data)")
plt.xlabel("lon"); plt.ylabel("lat"); plt.title(f"Habitat-exposure index — {len(risk)} sites")
savefig("05_habitat_exposure_index")

## Weight sensitivity: does the ranking survive other weightings?

In [ ]:
N_TRIALS = 500
rng = np.random.default_rng(SEED)

def random_weights(keys, rng):
    """Dirichlet gives random weights that sum to 1, so every trial is a valid weighting."""
    draws = rng.dirichlet(np.ones(len(keys)))
    return dict(zip(keys, draws))

base_rank = risk.set_index("site_id")["risk_score"].rank(ascending=False)
corrs, rank_matrix = [], []

for _ in range(N_TRIALS):
    vw = random_weights(list(VALUE_W), rng)
    ew = random_weights(list(EXPOSURE_W), rng)
    v = sum(value_norm[k] * w for k, w in vw.items())
    e = sum(exposure_norm[k] * w for k, w in ew.items())
    trial_score = np.sqrt((v + 1e-6) * (e + 1e-6))
    trial_rank = trial_score.rank(ascending=False)
    corrs.append(base_rank.corr(trial_rank.set_axis(static.site_id), method="spearman"))
    rank_matrix.append(trial_rank.values)

rank_matrix = np.array(rank_matrix)
rank_std = rank_matrix.std(axis=0)   # how much each site's rank jumps around across trials

print(f"Spearman correlation with the default-weight ranking, across {N_TRIALS} random weight draws:")
print(f"  mean {np.mean(corrs):.3f} | 5th pct {np.percentile(corrs, 5):.3f} | min {np.min(corrs):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].hist(corrs, bins=30, color="steelblue")
ax[0].set_title("How stable is the ranking under different weights?")
ax[0].set_xlabel("Spearman correlation vs. default weights"); ax[0].set_ylabel("count")

s = ax[1].scatter(static.lon, static.lat, c=rank_std, cmap="cividis", s=45)
plt.colorbar(s, ax=ax[1], label="rank std. dev. across trials")
ax[1].set_title("Which sites' rank is most sensitive to weight choice"); ax[1].set_xlabel("lon"); ax[1].set_ylabel("lat")
savefig("06_weight_sensitivity")